# Quickstart: direct normalisation (steps 1-2), one visit and a dataset
What a site (e.g. a hospital) runs on its own data with `Script/neonorm.py`: for every visit **step 1**, preprocessing, and **step 2**, the direct normalisation to the BCP templates of its age and to MNI, with QC. **Step 3**, the visit from its BCP template to MNI along the chain of BCP templates (`2.full_pipeline.ipynb`), can follow later and elsewhere: it needs only the outputs of step 2 (section 5) and the template edges.

| step | function | parts | main output (`<DATA_ROOT>/<id>/`) |
|---|---|---|---|
| 1 | `preprocess` | 1a orientation, 1b skull stripping (SynthStrip) + rigid T2 on the T1 grid, 1c QC | `<id>_T1w_brain_mask.nii.gz`, `<id>_T2w_rigid2T1.nii.gz`, `<id>_preprocess.json` |
| 2 | `direct_normalize` | 2a skull stripping + N4, 2b registration to each template of `direct_templates(age)`, 2c QC | `<id>_direct_to-<template>_*`, `<id>_direct_normalize.json`, `_qc.json` |

Sections: 0 setup, 1 the dataset table, 2-3 steps 1 and 2 for one visit, here in Python, 4 all visits on a Slurm cluster with the QC reviews, 5 what to keep for step 3. All options: `README.md`.

## 0. Setup
Once per computer: `bash Script/setup_neonorm.sh <software folder> <templates source>` (README, Installation; FSL 6.0.6.2 and FreeSurfer 8.0.0 set up first). It creates the conda environment `neonorm`, downloads ANTs 2.6.2, puts the SynthStrip models in one folder, copies and checks the templates, and writes the settings of the computer to `<software folder>/neonorm_env.sh`. Then, every time, `source <software folder>/neonorm_env.sh` in the shell **before** starting Jupyter, and the same line as `SETUP` of the Slurm job scripts. The cell below lists what is missing (empty: ready).

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from IPython.display import Image, display

REPO = Path('../..').resolve()  # this notebook is in <repository>/docs/tutorials
sys.path.insert(0, str(REPO / 'Script'))
os.environ.setdefault('NEONORM_BCP_TEMPLATES', str(REPO / 'templates' / 'BCP-atlas-for_release-Ver2.0.0'))
os.environ.setdefault('NEONORM_MNI_TEMPLATE', str(REPO / 'templates' / 'tpl-MNI152NLin2009cSym'))
os.environ.setdefault('NEONORM_NTHREADS', '8')  # CPU threads of ANTs in this notebook
import neonorm

SETUP = 'source /path/to/software/neonorm_env.sh'  # the setup line of every Slurm job script (setup_neonorm.sh)
EXAMPLE = Path('example').resolve()                 # absolute: the jsons record the paths
DATA_ROOT = EXAMPLE / 'out'                         # outputs, one folder per visit; review tables, Slurm logs
XFM_ROOT = REPO / 'templates' / 'tpl_xfm_build'     # template edges (tpl2tpl, avgsubj) for step 3

problems = neonorm.check_environment(verbose=True)
print('\nready' if not problems else '\nnot ready:\n  - ' + '\n  - '.join(problems))

## 1. The dataset: one row per visit
neonorm works on visits (one scan session), each in its own folder `<DATA_ROOT>/<id>/`, all files named `<id>_*`. Here the example visits, `example/participants.tsv`; for your own data write a table with the same columns:

| column | meaning |
|---|---|
| `participant_id` | the visit id and file prefix, e.g. `sub-001` (unique; a child with three visits has three ids) |
| `child` | the child; visits of one child share it (longitudinal analyses) |
| `age_months` | age at scan in months (chooses the SynthStrip models, the volume check and the BCP templates) |
| `t1` | the T1w (NIfTI, whole head, not skull-stripped); only read, never changed |
| `t2` | the T2w of the same session (optional, empty if none): rigid on the T1 grid in step 1, used for the SUITPy cerebellum of the QC |

**Example data**: `sub-ex01`, an infant (BCP, 5.6 months), and `sub-ex02`, a child (HCPD, 8 years). They are NDA data and stay local (`example/` is not in the repository).

In [ ]:
visits = pd.read_csv(EXAMPLE / 'participants.tsv', sep='\t', keep_default_na=False)
for c in ('t1', 't2'):  # absolute paths (relative ones are relative to the table)
    visits[c] = [str(EXAMPLE / f) if f else '' for f in visits[c]]
visits['out_dir'] = [str(DATA_ROOT / i) for i in visits['participant_id']]
assert visits['participant_id'].is_unique
DATA_ROOT.mkdir(parents=True, exist_ok=True)
visits

## 2. Step 1: preprocessing (`preprocess`)
`preprocess(t1, age_months, out_dir, t2=None, prefix=None)`; command line `neonorm.py <t1> <age_months> <out_dir> [--t2 <t2>] [--prefix <id>]`

**Input**: the T1 and (optional) T2 as acquired (whole head, any orientation); the age at scan in months.

| part | function | what | output (`<out_dir>`) |
|---|---|---|---|
| 1a | `orient` | orientation: `fslreorient2std` (FSL) only if an image is not in the standard orientation (lossless, world coordinates unchanged) | `<id>_T1w_reorient.nii.gz`, `_T2w_reorient.nii.gz` (only if reoriented) |
| 1b | `skull_strip_step` | skull stripping: SynthStrip brain mask (FreeSurfer `mri_synthstrip`; below 12 months the union of the default and the infant model, from 12 months the default model); with a T2: the T2 registered rigidly to the T1 (ANTsPy, whole head), on the T1 grid | `<id>_T1w_brain_mask.nii.gz`; `<id>_T2w_rigid2T1.nii.gz`, `_0GenericAffine.mat` |
| 1c | `skull_strip_step` (QC) | automatic QC: SUITPy cerebellum of the T1, run twice (the runs agree: Dice >= 0.95; at least 95 % of it inside the brain mask); brain volume against the BCP visits of the same age (robust z, &#124;z&#124; <= 3) | `needs_visual_check`; `<id>_skullstrip_qc.png` (flagged visits, or always with `always_qc_figure`) |

**Record**: `<id>_preprocess.json`: inputs, software, `orient`, `skull_strip` (files and QC). **Step 2 uses** `orient.t1`, `skull_strip.brain_mask`, `skull_strip.t2_on_t1`. **Resources**: 2 CPUs, 24 GB (SUITPy), 3-5 minutes.

### 2.1 One visit (here in Python)

In [ ]:
V = visits.iloc[0]  # sub-ex01 (infant); visits.iloc[1]: sub-ex02 (child)
pipe = neonorm.preprocess(V.t1, V.age_months, V.out_dir, t2=V.t2 or None, prefix=V.participant_id,
                          always_qc_figure=True)
print(pipe['skull_strip']['brain_mask_rule'])
display(pd.Series(pipe['skull_strip']['qc']))
display(Image(pipe['skull_strip']['qc_figure']))

## 3. Step 2: direct normalisation (`direct_normalize`)
`direct_normalize(t1, age_months, brain_mask, out_dir, t2=None, prefix=None, steps=(1, 2, 3), templates=None)`; command line `neonorm.py --direct_normalize <t1> <age_months> <out_dir> --brain_mask <mask> [--t2 <rigid T2>] --templates <names>`

**Input**: from step 1 the T1 (`orient.t1`, not skull-stripped), the brain mask and the rigid T2 (optional); the age; `templates`: here always `direct_templates(age)`, the BCP templates of the visit's age window and MNI (below).

| part (`steps`) | function | what | output (`<out_dir>`) |
|---|---|---|---|
| 2a (1) | `head2brain_and_n4correction` | skull stripping (ANTs `ImageMath m`: image x brain mask) and bias correction (`N4BiasFieldCorrection`, inside the brain mask) of the T1 and T2 | `<id>_T1w_brain_n4.nii.gz`, `_T2w_brain_n4.nii.gz` |
| 2b (2) | `register_to_template` | registration of the N4 T1 to each template: `antsRegistrationSyN.sh` default (rigid + affine + SyN), T1 only, starting from a rotation search (`antsAI`) against the nearest BCP template | per template `<id>_direct_to-<template>_0GenericAffine.mat`, `_1Warp.nii.gz` (visit -> template: `[warp, affine]`), `_1InverseWarp.nii.gz` (template -> visit), `_Warped.nii.gz`, `_init.mat` |
| 2c (3) | `direct_normalization_qc` | the reference cerebellum of the visit (`brain_to_cerebellum`: SUITPy on the N4 T1 with the T2 and the brain mask, 2-3 runs, the medoid; chosen without any registration); per template: T1 r, brain and cerebellum Dice and HD95, Jacobian; `needs_visual_check`: cerebellum Dice with the main (nearest) template below 0.90 | `<id>_cerebellum_suitpy_from-brain-N4_dseg.nii.gz` (runs in `cerebellum_suitpy/`); `<id>_direct_to-<template>_qc.png`; `<id>_direct_normalize_qc.json` |

**Templates** (`direct_templates(age)`): the BCP template of the nearest atlas month (first: the main template of the QC, and the start of the chains of step 3), the other templates of the visit's avgsubj edges (`avgsubj_edges(age)`: usually the two atlas months around the age, three at an atlas month; from 60 months BCP60M only), and MNI. E.g. 5.55 months: BCP06M, BCP05M, BCP07M, MNI; 97 months: BCP60M, MNI. With these, every way of step 3 can be run later, and the visit can join the avgsubj averages of its age window.

**Record**: `<id>_direct_normalize.json` (2a-2b: inputs, software, the N4 images, per template the files and the `antsRegistration` calls). **Step 3 uses** this json and the reference cerebellum. Any part can run again alone, e.g. `steps=(3,)`: the QC. **Resources**: 2a-2b 8 CPUs, 16 GB, 7-14 minutes per template; 2c 4 CPUs, 24 GB (SUITPy); about 0.5 GB per template (the warps).

### 3.1 One visit (here in Python)

In [ ]:
templates = neonorm.direct_templates(V.age_months)
print(V.participant_id, V.age_months, 'months:', templates)
norm = neonorm.direct_normalize(pipe['orient']['t1'], V.age_months, pipe['skull_strip']['brain_mask'], V.out_dir,
                                t2=pipe['skull_strip'].get('t2_on_t1'), prefix=V.participant_id, templates=templates)
qc = pd.DataFrame(norm['qc']['qc']).T
display(qc[['t1_corr', 'brain_dice', 'brain_log10_hd95', 'cerebellum_dice', 'cerebellum_log10_hd95', 'jacobian_min']])
print('needs_visual_check:', norm['qc']['needs_visual_check'])
for name in norm['normalize']:
    display(Image(Path(V.out_dir) / f'{V.participant_id}_direct_to-{name}_qc.png'))

## 4. All visits (Slurm)

### 4.1 Step 1, all visits
One array task per visit, the command line of `neonorm.py` (it prints every command and its output into the task's log). The cell writes the task list and the array script; `SUBMIT_STEP1 = True` submits it.

In [ ]:
log_dir = DATA_ROOT / 'logs' / 'step1'
log_dir.mkdir(parents=True, exist_ok=True)
tasks = log_dir / 'tasks.tsv'  # t2 last: an empty field there is safe for read
visits[['participant_id', 't1', 'age_months', 'out_dir', 't2']].to_csv(tasks, sep='\t', index=False, header=False)
script = log_dir / 'step1_array.sh'
script.write_text(f"""#!/bin/bash
#SBATCH --job-name=step1
#SBATCH --array=1-{len(visits)}%50
#SBATCH --cpus-per-task=2
#SBATCH --mem=24G
#SBATCH --time=01:00:00
#SBATCH --output={log_dir}/step1_%A_%a.out
{SETUP}
IFS=$'\\t' read -r prefix t1 age out t2 < <(sed -n "${{SLURM_ARRAY_TASK_ID}}p" {tasks})
python {REPO / 'Script' / 'neonorm.py'} "$t1" "$age" "$out" --prefix "$prefix" ${{t2:+--t2 "$t2"}}
""")
print(script.read_text())
SUBMIT_STEP1 = False  # True: submit
if SUBMIT_STEP1:
    print(subprocess.run(['sbatch', str(script)], capture_output=True, text=True).stdout)

### 4.2 QC review of step 1 → `skullstrip_usable`
One row per visit from `<id>_preprocess.json`. Flagged visits (`needs_visual_check`: the two SUITPy runs disagree, the cerebellum is not inside the mask, or the brain volume is unusual for the age) need a look at their figure; write the decision per visit (`yes` / `no`) into `skullstrip_review.tsv`. Step 2 runs only for `yes`.

In [ ]:
rows = []
for v in visits.itertuples():
    f = Path(v.out_dir) / f'{v.participant_id}_preprocess.json'
    if not f.exists():
        rows.append({'participant_id': v.participant_id, 'status': 'not run or failed'})
        continue
    s = json.loads(f.read_text())['skull_strip']
    qc = s.get('qc', {})
    rows.append({'participant_id': v.participant_id, 'status': 'done', 'brain_mask_rule': s['brain_mask_rule'],
                 'suitpy_runs_dice': qc.get('suitpy_runs_dice'), 'volume_z': qc.get('volume_z'),
                 'needs_visual_check': qc.get('needs_visual_check'), 'flags': ' | '.join(qc.get('flags', [])),
                 'qc_figure': s.get('qc_figure', '')})
ss_qc = pd.DataFrame(rows).set_index('participant_id')
# decision: yes unless the figure shows a wrong mask (edit the column of the flagged visits after looking)
ss_qc['skullstrip_usable'] = np.where(ss_qc['status'] == 'done', 'yes', 'no')
ss_qc.to_csv(DATA_ROOT / 'skullstrip_review.tsv', sep='\t')
for i, r in ss_qc[ss_qc['needs_visual_check'] == True].iterrows() if 'needs_visual_check' in ss_qc else []:
    print(i, '-', r['flags'])
    display(Image(r['qc_figure']))
ss_qc

### 4.3 Step 2, all visits
`submit_direct_normalize_array` submits two job arrays, `direct` (2a-2b, one task per visit) and `directqc` (2c, task i after task i of `direct` succeeded), at most `max_running` tasks at once; per visit the job scripts and full logs in `<out_dir>/log/`. Only visits with `skullstrip_usable == yes`, each to `direct_templates(age)`.

In [ ]:
def step2_args(v):
    """Arguments of direct_normalize for a visit after step 1 (from its preprocess.json), with its templates."""
    log = json.loads((Path(v.out_dir) / f'{v.participant_id}_preprocess.json').read_text())
    return {'t1': log['orient']['t1'], 'age_months': float(v.age_months), 'brain_mask': log['skull_strip']['brain_mask'],
            'out_dir': v.out_dir, 't2': log['skull_strip'].get('t2_on_t1'), 'prefix': v.participant_id,
            'templates': neonorm.direct_templates(v.age_months)}


SUBMIT_STEP2 = False  # True: submit
use = visits[visits['participant_id'].map(pd.read_csv(DATA_ROOT / 'skullstrip_review.tsv', sep='\t',
                                                       index_col='participant_id')['skullstrip_usable']) == 'yes']
print(f'{len(use)} of {len(visits)} visits')
if SUBMIT_STEP2:
    arr = neonorm.submit_direct_normalize_array([step2_args(v) for v in use.itertuples()],
                                                DATA_ROOT / 'logs' / datetime.now().strftime('step2_%Y%m%d_%H%M%S'),
                                                steps=(1, 2, 3), setup=SETUP, max_running=50)
    print('arrays', arr['job_id'], arr['qc_job_id'])

### 4.4 QC review of step 2 → `direct_usable` per registration
One row per visit and template from `<id>_direct_normalize_qc.json` (`role`: `nearest` = the main template, `MNI`, `other` = the other BCP templates of the age window). `needs_visual_check`: the cerebellum Dice with the nearest template is below `CEREB_DICE_MIN` (0.90). Look at the figures of the flagged registrations (template above, warped T1 below; template mask and cerebellum as lines) and of those far from the others of the same age (low T1 r, high log10 HD95), then write `direct_usable` (`yes` / `no`) per row into `direct_review.tsv`. Step 3 uses: `direct` the MNI registration, `tpl2tpl` and `avgsubj` the registration to the nearest BCP template. Direct registration to MNI often fails below about a year (project: notebook 3, 3.4-3.5).

In [ ]:
rows = []
for v in visits.itertuples():
    f = Path(v.out_dir) / f'{v.participant_id}_direct_normalize_qc.json'
    if not f.exists():
        continue
    q = json.loads(f.read_text())
    for tpl, m in q['qc'].items():
        rows.append({'participant_id': v.participant_id, 'age_months': v.age_months, 'template': tpl,
                     'role': 'nearest' if tpl == q['main_template'] else ('MNI' if tpl == neonorm.MNI_NAME else 'other'),
                     **{k: m.get(k) for k in ('t1_corr', 'brain_dice', 'brain_log10_hd95', 'cerebellum_dice',
                                              'cerebellum_log10_hd95', 'jacobian_min')},
                     'needs_visual_check': q['needs_visual_check'], 'qc_figure': m.get('qc_figure')})
direct_qc = pd.DataFrame(rows)
if len(direct_qc):
    # decision: yes unless a figure shows a failed registration (edit the column after looking)
    direct_qc['direct_usable'] = 'yes'
    direct_qc.to_csv(DATA_ROOT / 'direct_review.tsv', sep='\t', index=False)
    for r in direct_qc[direct_qc['needs_visual_check']].itertuples():
        print(r.participant_id, r.template)
        display(Image(r.qc_figure, width=900))
direct_qc.round(3)

## 5. What to keep for step 3
Step 3 (`2.full_pipeline.ipynb`) needs per visit, in its folder: `<id>_direct_normalize.json` and `_qc.json`, `<id>_T1w_brain_n4.nii.gz`, `<id>_T1w_brain_mask.nii.gz`, the reference cerebellum `<id>_cerebellum_suitpy_from-brain-N4_dseg.nii.gz`, and per template `<id>_direct_to-<template>_0GenericAffine.mat`, `_1Warp.nii.gz` and `_1InverseWarp.nii.gz` (the inverse warps: for the avgsubj averages, template a -> visit -> template b); and the two review tables. `_Warped.nii.gz` and the `_qc.png` figures are for the review only.

The jsons record absolute paths: when the visit folders are copied to another computer for step 3, the paths in the jsons have to point to the new place (open: a function for this).